# 🤖 Transformer Playground

Here we implement a miniature Self-Attention block from scratch in PyTorch to demystify the Q, K, V matrices.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### Self-Attention Implementation

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        
        assert (self.head_dim * heads == embed_size), "Embed size needs to be div by heads"
        
        self.values = nn.Linear(embed_size, embed_size, bias=False)
        self.keys = nn.Linear(embed_size, embed_size, bias=False)
        self.queries = nn.Linear(embed_size, embed_size, bias=False)
        self.fc_out = nn.Linear(embed_size, embed_size)
        
    def forward(self, values, keys, queries, mask=None):
        N = queries.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], queries.shape[1]
        
        # Pass through linear layers
        v = self.values(values).view(N, value_len, self.heads, self.head_dim)
        k = self.keys(keys).view(N, key_len, self.heads, self.head_dim)
        q = self.queries(queries).view(N, query_len, self.heads, self.head_dim)
        
        # Dot product (Q * K^T)
        energy = torch.einsum("nqhd,nkhd->nhqk", [q, k])
        
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))
            
        attention = torch.softmax(energy / math.sqrt(self.head_dim), dim=3)
        
        # Multiply by V
        out = torch.einsum("nhqk,nvhd->nqhd", [attention, v]).reshape(N, query_len, self.embed_size)
        
        return self.fc_out(out), attention

print("Self-Attention Module Created!")

In [ ]:
# Test the module
embed_size = 256
heads = 8
seq_length = 10
batch_size = 2

x = torch.randn(batch_size, seq_length, embed_size)
attention_layer = SelfAttention(embed_size, heads)

out, attention_weights = attention_layer(x, x, x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention Weights shape: {attention_weights.shape} -> (Batch, Heads, QueryLen, KeyLen)")